# Kata 25 — Gestión de Sesiones (Resume y Fork)

> Spec: [`specs/025-session-management/spec.md`](../../specs/025-session-management/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

Resume / fork son operaciones del Claude Code CLI / Agent SDK. En este notebook simulo el patrón "fresh session con summary inyectado" usando la API directa, que es el caso más práctico (y el más confiable cuando el contexto previo puede estar stale).

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import pathlib, json

client, settings = bootstrap(budget_calls=6)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Tres patrones para preservar contexto:

- `--resume <name>`: continúa sesión nombrada. Sólo si los tool results son
  todavía válidos.
- `fork_session`: rama paralela desde una baseline.
- Sesión nueva con summary inyectado: cuando el mundo cambió y los tool
  results previos están stale.

## 2. Por qué importa

Resumir una sesión vieja con archivos modificados → el modelo razona
sobre archivos que ya no son lo que cree.

## 3. Patrón "fresh con summary inyectado" (más confiable)

In [ ]:
# Sesión 1: investigación inicial. Persiste hallazgos en scratchpad.
SCRATCHPAD = pathlib.Path("./investigation-summary.md")

resp1 = client.messages.create(
    system="Eres un investigador. Cuando llegues a una conclusión, escribe línea exacta: SUMMARY: ... ",
    messages=[{"role":"user","content":"Considera que el bug está en src/auth/login.py línea 42. ¿Qué cambio mínimo lo arreglaría? Empieza tu última línea con SUMMARY:"}],
)
text1 = next((b.text for b in resp1.content if b.type == "text"), "")
print("=== sesión 1 ===")
print(text1[:400])

# Extraer summary
for line in text1.splitlines():
    if line.strip().startswith("SUMMARY:"):
        SCRATCHPAD.write_text(line.strip())
        break

print("\nscratchpad:", SCRATCHPAD.read_text())

In [ ]:
# Sesión 2: el repo cambió desde la sesión 1. NUEVA sesión con summary inyectado,
# en lugar de --resume con tool results stale.

summary = SCRATCHPAD.read_text() if SCRATCHPAD.exists() else "(no summary)"

resp2 = client.messages.create(
    system=(
        "Continúas una investigación previa. Resumen de hallazgos:\n\n"
        f"{summary}\n\n"
        "El repo fue modificado: src/auth/login.py ahora tiene una validación adicional "
        "en línea 50. Aplica el cambio del summary tomando esto en cuenta."
    ),
    messages=[{"role":"user","content":"¿Cómo procedo?"}],
)
print("=== sesión 2 (fresh + summary) ===")
print(next((b.text for b in resp2.content if b.type == "text"), "")[:500])

### 3.1 Decisión rápida

In [ ]:
resp_decide = client.messages.create(
    system="""Eres un experto en Claude Code. Para cada caso responde con una palabra:
'resume', 'fork', o 'fresh-with-summary', y la razón en una línea.""",
    messages=[{"role":"user","content":"""Decide para cada caso:

1. Mi compañero modificó 30 archivos del repo. Necesito continuar la investigación.
2. Quiero comparar dos enfoques de refactor desde el mismo punto.
3. La sesión continúa lógicamente y los archivos no han cambiado.
4. Necesito retomar análisis de hace 3 días pero el feature está casi reescrito.
"""}],
)
print(next((b.text for b in resp_decide.content if b.type == "text"), ""))

## 4. Anti-patrón — `--resume` con tool results stale

No puedo demostrarlo con la API directa (sin sesiones nombradas), pero el síntoma es claro:

- Sesión 1: el agente leyó `src/auth.py` (versión A).
- Tras `--resume`, el agente "recuerda" `auth.py` como versión A.
- Realidad: el archivo se reescribió, ahora es versión B.
- Resultado: el agente da consejos basados en código que ya no existe.

**Defensa**: cuando hay incertidumbre sobre si los archivos cambiaron,
arrancar **fresh** con el summary del Kata 18 inyectado en el system
prompt.

## 5. Argumento de certificación

- **Resume**: contexto válido, conversación continúa lógicamente.
- **Fork**: dos caminos paralelos desde una baseline.
- **Fresh + summary**: tool results envejecieron; mejor empezar limpio.
- Conexión: el summary del scratchpad (Kata 18) es la entrada del
  patrón fresh.

## 6. Auto-evaluación

**1. Mi compañero modificó 30 archivos. ¿Resume o fresh?**

Fresh con summary. Los tool results de la sesión vieja (que leyó archivos
versión A) ahora están stale.

**2. Quiero comparar dos refactorizaciones desde la misma baseline.**

`fork_session`. Cada rama explora un enfoque sin interferir con la otra.

**3. ¿Cuándo `--resume` falla silenciosamente?**

Cuando el modelo asume validez de tool results que ya no la tienen. El
síntoma típico: el modelo cita un archivo o función que ya no existe.